# Probabilistic transformer head study

This notebook compares different output heads for the Probabilistic Transformer:
- Gaussian
- Johnson SU
- Mixture of Gaussians (1, 2, 3, 5 components)
- Mixture of Johnson SU (1, 2, 3, 5 components)
- Quantile Regression (9, 19, ..., 99 quantiles)

Each configuration is run 10 times to account for statistical variability.

## Metrics
- **Point**: MAE, RMSE, MAPE
- **Probabilistic**: PICP, MPIW, PINAW, Interval Score, CRPS
- **Quantile**: Pinball losses (10th, 50th, 90th percentiles)

In [7]:
import os
import sys
import gc
import time
from pathlib import Path
from typing import Dict, Any, List

import numpy as np
import pandas as pd
import tensorflow as tf

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ExperimentConfig, DataConfig, ModelConfig, TrainingConfig
from models import ProbabilisticTransformer
from core.trainer import Trainer
from core.evaluator import Evaluator
from core.experiment_utils import (
    CANONICAL_MODEL_CONFIG, CANONICAL_TRAIN_CONFIG, N_RUNS,
    load_data, load_cache, save_cache, set_seeds, save_run, save_summary,
)

try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

Project root: /home/d1ff1cult/masterproef_new


In [8]:
RESULTS_DIR = Path(project_root) / "results" / "head_study"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "head_study_results.json"

configs = []

configs.append({"head_type": "gaussian", "head_params": {}, "name": "Gaussian"})
configs.append({"head_type": "johnson_su", "head_params": {}, "name": "JohnsonSU"})

for n in [1, 2, 3, 5]:
    configs.append({"head_type": "mixture_gaussian", "head_params": {"n_components": n}, "name": f"GMM-{n}"})
    configs.append({"head_type": "mixture_johnson_su", "head_params": {"n_components": n}, "name": f"JSU-Mix-{n}"})

for n in [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]:
    q_list = np.linspace(1/(n+1), n/(n+1), n).tolist()
    configs.append({"head_type": "quantile", "head_params": {"quantiles": q_list}, "name": f"Quantile-{n}"})

print(f"Total configurations: {len(configs)}")
print(f"Total training runs: {len(configs) * N_RUNS}")

Total configurations: 20
Total training runs: 200


In [9]:
data = load_data()

Train: (10367, 168, 28), Val: (1997, 168, 28)


In [10]:
results_cache = load_cache(CACHE_FILE)

In [11]:
for i, conf in enumerate(configs):
    conf_name = conf["name"]
    print(f"\n[{i+1}/{len(configs)}] Processing {conf_name}...")

    if conf_name not in results_cache:
        results_cache[conf_name] = {"runs": []}

    completed_runs = len(results_cache[conf_name]["runs"])
    if completed_runs >= N_RUNS:
        print(f"  Already completed {completed_runs} runs. Skipping.")
        continue

    runs_to_do = N_RUNS - completed_runs
    exp_results_dir = RESULTS_DIR / conf_name.replace(" ", "_")
    print(f"  Running {runs_to_do} repetitions...")

    for run_idx in range(runs_to_do):
        current_run_id = completed_runs + run_idx + 1
        print(f"    Run {current_run_id}/{N_RUNS}...")

        tf.keras.backend.clear_session()
        gc.collect()
        set_seeds(current_run_id)

        exp_config = ExperimentConfig(
            name=f"{conf_name}_run_{current_run_id}",
            data_config=data.data_config,
            model_config=ModelConfig(**CANONICAL_MODEL_CONFIG),
            training_config=TrainingConfig(**CANONICAL_TRAIN_CONFIG),
            head_type=conf["head_type"],
            head_params=conf["head_params"],
        )

        try:
            model = ProbabilisticTransformer(exp_config)
            trainer = Trainer(exp_config)

            t0 = time.time()
            trainer.train(model, data.X_train_s, data.y_train_s,
                          data.X_val_s, data.y_val_s)
            training_time = time.time() - t0

            evaluator = Evaluator(model, data.scaler)
            metrics = evaluator.evaluate(data.X_test_s, data.y_test)
            metrics["training_time"] = training_time

            y_pred_mean = evaluator.generate_forecasts(data.X_test_s)
            q_originals = evaluator.generate_quantile_forecasts(data.X_test_s)
            predictions = {"y_pred_mean": y_pred_mean}
            for q, arr in q_originals.items():
                predictions[f"q_{q}"] = arr

            save_run(exp_results_dir, current_run_id - 1, model, metrics, predictions, data.y_test)

            run_result = {
                "run_id": current_run_id,
                "metrics": {k: float(v) if v is not None and np.isfinite(v) else None for k, v in metrics.items()},
                "fit_time_s": training_time,
            }
            results_cache[conf_name]["runs"].append(run_result)
            save_cache(CACHE_FILE, results_cache)

            print(f"      MAE: {metrics.get('MAE', 0):.4f}")

        except Exception as e:
            print(f"      FAILED: {e}")
            import traceback
            traceback.print_exc()



[1/20] Processing Gaussian...
  Already completed 10 runs. Skipping.

[2/20] Processing JohnsonSU...
  Already completed 10 runs. Skipping.

[3/20] Processing GMM-1...
  Already completed 10 runs. Skipping.

[4/20] Processing JSU-Mix-1...
  Already completed 10 runs. Skipping.

[5/20] Processing GMM-2...
  Already completed 10 runs. Skipping.

[6/20] Processing JSU-Mix-2...
  Already completed 10 runs. Skipping.

[7/20] Processing GMM-3...
  Already completed 10 runs. Skipping.

[8/20] Processing JSU-Mix-3...
  Already completed 10 runs. Skipping.

[9/20] Processing GMM-5...
  Already completed 10 runs. Skipping.

[10/20] Processing JSU-Mix-5...
  Already completed 10 runs. Skipping.

[11/20] Processing Quantile-9...
  Already completed 10 runs. Skipping.

[12/20] Processing Quantile-19...
  Already completed 10 runs. Skipping.

[13/20] Processing Quantile-29...
  Already completed 10 runs. Skipping.

[14/20] Processing Quantile-39...
  Already completed 10 runs. Skipping.

[15/20] Pr

In [12]:
# Analysis
# Probabilistic metrics (use .get() for backward compatibility with cached runs)
PROB_METRICS = ["MAPE", "PICP", "MPIW", "PINAW", "IntervalScore", "CRPS", "Pinball_10", "Pinball_50", "Pinball_90", "Avg_Pinball"]

summary_rows = []

for name, data in results_cache.items():
    runs = data["runs"]
    if not runs:
        continue

    maes = [r["metrics"]["MAE"] for r in runs]
    rmses = [r["metrics"]["RMSE"] for r in runs]
    times = [r["fit_time_s"] for r in runs]

    row = {
        "Configuration": name,
        "MAE_mean": np.mean(maes),
        "MAE_std": np.std(maes),
        "RMSE_mean": np.mean(rmses),
        "FitTime_mean": np.mean(times),
    }
    for m in PROB_METRICS:
        vals = [r["metrics"].get(m) for r in runs]
        vals = [v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
        row[f"{m}_mean"] = np.mean(vals) if vals else np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("MAE_mean")
display(summary_df)

,Configuration,MAE_mean,MAE_std,RMSE_mean,FitTime_mean,MAPE_mean,PICP_mean,MPIW_mean,PINAW_mean,IntervalScore_mean,CRPS_mean,Pinball_10_mean,Pinball_50_mean,Pinball_90_mean,Avg_Pinball_mean
9,JSU-Mix-5,20.486906,0.899982,27.838903,76.864938,5557.336614,0.912755,96.199056,0.300209,150.299702,15.233428,5.600228,10.374968,4.396024,6.790407
8,GMM-5,20.546982,0.769989,27.923528,77.404056,5158.307362,0.920346,97.147817,0.303170,147.435309,15.314535,5.516064,10.474572,4.453882,6.814839
5,JSU-Mix-2,20.584979,1.258706,28.035078,75.192348,5453.884735,0.922957,100.056090,0.312246,149.982258,15.120831,5.505176,10.284218,4.442106,6.743833
4,GMM-2,20.722178,0.776770,27.925356,76.887846,5460.894080,0.928598,100.455712,0.313493,144.017843,15.376189,5.289919,10.543380,4.631953,6.821750
1,JohnsonSU,20.722283,0.813122,27.977882,69.817041,NaN,0.934641,109.171123,NaN,149.957796,15.173047,NaN,NaN,NaN,NaN
3,JSU-Mix-1,20.816882,0.720998,28.095221,72.775465,6074.386311,0.930319,107.181447,0.334482,149.715384,15.141500,5.449450,10.287762,4.502358,6.746523
6,GMM-3,20.836685,1.126780,28.150307,73.210822,5528.992534,0.935468,104.847702,0.327199,147.882445,15.605825,5.431497,10.690058,4.662582,6.928046
7,JSU-Mix-3,20.850704,1.241572,28.314749,76.587209,5878.516734,0.895342,91.249746,0.284764,158.194601,15.490767,5.874319,10.499688,4.425137,6.933048
19,Quantile-99,21.048326,0.685330,28.659061,81.463254,5728.845179,0.907233,97.036753,0.302823,157.822089,15.554108,5.914029,10.524163,4.528287,6.988827
10,Quantile-9,21.179024,1.160084,28.686258,84.096354,5750.149575,0.804599,73.226263,0.228518,210.326026,15.965822,6.051603,10.589512,4.638252,7.093122
